<a href="https://colab.research.google.com/github/jooyongsim/deeplearning_SMWU_2026/blob/main/2026%EB%94%A5%EB%9F%AC%EB%8B%9D_Lab5_pusht_imitation_min_compact.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Imitation Learning PushT — 사람의 시연으로 policy 학습하기 (Mini PushT Simulator)
## by Joo Yong Sim (2026.10.19) @ Sookmyung W. Univ, Mechanical System Eng

이 튜토리얼에서 개념을 학습하고 아래 튜토리얼에서 실제 모방학습을 해봅시다.

- `pusht_mini_sim.py` : 물리 (impulse, friction, restitution) 를 직접 구현했고
- `pusht_keyboard_sim.py` : **사람이 policy 역할**을 하며 키보드로 T를 밀어

목표는 사람이 한 일을 **신경망이 따라 하도록** 만듭니다. 이것이 imitation learning (모방 학습) 입니다.

$$
\boxed{
\underbrace{\text{사람 시연}}_{\mathcal D=\{(o_t,a_t)\}}
\;\rightarrow\;
\underbrace{\text{데이터셋}}_{(o,\ \mathbf a_{t:t+H})}
\;\rightarrow\;
\underbrace{\text{학습}}_{\pi_\phi(\mathbf a\mid o)}
\;\rightarrow\;
\underbrace{\text{closed-loop 평가}}_{\text{success rate}}
}
$$

**이 튜토리얼에서 답할 두 가지 질문**

1. **시연(demonstration)을 몇 개나 모아야 하나?** → 6절 (문헌 기준) + 13절 (직접 실험)
2. **어떤 모델을 써야 하나?** → 8절 (multimodality 실험) + 9–12절 (MLP-BC vs Diffusion Policy)

**전체 단계**

| 절 | 내용 |
|---|---|
| 1–2 | 물리 불러오기, 랜덤 초기 위치 + 랜덤 target pose 환경 |
| 3 | 시연 데이터 형식 (episode, `.npz`) |
| 4 | **사람 시연 수집 GUI** (키보드 / 마우스) |
| 5 | Scripted expert — 강의의 impulse 식으로 만든 "가상의 사람" (빠른 대량 수집용) |
| 6 | 몇 개나 모아야 하나? |
| 7 | 데이터셋: goal 좌표계, observation history, action chunk, 정규화 |
| 8 | 어떤 모델? — multimodality 장난감 실험 |
| 9–10 | Model 1: MLP Behavior Cloning, Model 2: Diffusion Policy |
| 11–12 | 학습, closed-loop 평가 |
| 13 | 데이터 수 vs 성공률 실험 |
| 14 | 학습된 policy를 창에서 보기, 해 볼 것 |

**실행 방법**

- 전체 튜토리얼: `python pusht_imitation.py` (그림은 `figures_imitation/` 에 저장)
- **사람 시연 수집만**: `python pusht_imitation.py collect` → 키보드 창이 열림
- VS Code Interactive / Jupyter: 위에서부터 셀 실행. 4절의 `COLLECT_HUMAN = True` 로 바꾸면 수집 창이 열립니다.
- notebook으로 바꾸기: `python ../tools/py2ipynb.py pusht_imitation.py`

필요한 것: `numpy`, `matplotlib`, `torch` (CPU로 충분, GPU가 있으면 자동 사용), 그리고 같은 폴더의 `pusht_keyboard_sim.py`.



강의(Engineering Dynamics)에서 다룬 식들이 그대로 등장합니다.

— Pymunk 없이 직접 구현하는 2D rigid-body contact.

목표: $\boxed{\text{action}\rightarrow\text{pusher movement}\rightarrow\text{collision}\rightarrow J\rightarrow(v,\omega)\rightarrow(x,\theta)}$ 연결 고리가 **실제 코드에서 어떻게 이어지는지** 보기.

| 강의 내용 (Engineering Dynamics) | 이 코드에서 |
|---|---|
| Mass moment of inertia, parallel-axis theorem | `make_T_geometry()` |
| Relative velocity $\mathbf v_C = \mathbf v_G + \boldsymbol\omega\times\mathbf r$ · Impulse–momentum (linear + angular) | `resolve_impulse()` |
| Coefficient of restitution $e$ · Coulomb friction $\lvert j_t\rvert \le \mu j_n$ | `restitution`, `friction` |

## 1. PushT에서 1 step은 정확히 무엇인가?

Policy가 action $a_t=(x_{\text{target}},\,y_{\text{target}})$ 을 하나 출력하면, 한 control step 안에서 physics를 여러 번 계산합니다.

```text
action(target pos) → PD controller → pusher vel → pusher pos → circle↔T collision detection
  → (contact point, normal, penetration) → impulse J → T의 v, ω 변경 → x, y, θ 적분
```

예: control 10 Hz, physics 100 Hz 이면 $\Delta t_{\text{control}}=0.1,\ \Delta t_{\text{physics}}=0.01$ → 하나의 `step()`에서 physics 10번.
아래 코드는 `numpy`만 있으면 실행됩니다 (그림은 `matplotlib`).

In [ ]:
import math
from dataclasses import dataclass
import numpy as np

## 2. Basic 2D math

2D 각속도는 $z$ 성분 하나뿐: $\boldsymbol\omega=(0,0,\omega)$.

$$
\underbrace{\mathbf a\times\mathbf b = a_x b_y - a_y b_x}_{\text{2D cross (scalar)}}\qquad
\underbrace{(0,0,\omega)\times(r_x,r_y,0) = (-\omega r_y,\ \omega r_x)}_{\boldsymbol\omega\times\mathbf r}\qquad
\underbrace{R(\theta)=\begin{bmatrix}\cos\theta & -\sin\theta\\ \sin\theta & \cos\theta\end{bmatrix}}_{\text{local}\rightarrow\text{world}}
$$

In [ ]:
# ============================================================
# 1. Basic 2D math
# ============================================================
def rotation_matrix(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s], [s, c]], dtype=float)

def cross2(a, b):
    """2D vector cross product: a x b -> scalar"""
    return float(a[0] * b[1] - a[1] * b[0])

def omega_cross_r(omega, r):
    """[0,0,omega] x [rx, ry, 0] = [-omega*ry, omega*rx]"""
    return np.array([-omega * r[1], omega * r[0]])

## 3. T polygon + mass moment of inertia

T를 **top rectangle + stem rectangle** 으로 나누어 COM과 관성모멘트를 구하고, collision은 8개 꼭짓점의 concave T polygon으로 처리합니다.

$$
y_{\text{com}}=\frac{A_{\text{top}}\,y_{\text{top}}+A_{\text{stem}}\,y_{\text{stem}}}{A_{\text{top}}+A_{\text{stem}}}\qquad
I_c=\frac{m\,(w^2+h^2)}{12}\qquad
\underbrace{I = I_c + m\,d^2}_{\text{parallel-axis}},\quad I_{\text{T}} = I_{\text{top}} + I_{\text{stem}}
$$

마지막으로 꼭짓점을 $y_{\text{com}}$ 만큼 내려 **body local origin = COM** 으로 만듭니다 → 이후 $\mathbf r$ (COM → contact) 를 바로 사용.

In [ ]:
# ============================================================
# 2. Make T polygon + moment of inertia
# ============================================================
def make_T_geometry(W=1.8, H=1.8, stem_w=0.5, top_t=0.5, mass=1.0):
    """
    T shape:  ┌────────────┐   T를 top rectangle + stem rectangle 로 생각해서
              └───┐    ┌───┘   COM / inertia 계산.
                  │stem│       collision 자체는 아래의 concave T polygon으로 처리.
                  └────┘
    """
    # areas
    top_area = W * top_t
    stem_h = H - top_t
    stem_area = stem_w * stem_h
    total_area = top_area + stem_area

    # rectangle centers  →  center of mass
    y_top = H / 2 - top_t / 2
    y_stem = -top_t / 2
    y_com = (top_area * y_top + stem_area * y_stem) / total_area

    # mass split
    m_top = mass * top_area / total_area
    m_stem = mass * stem_area / total_area

    # inertia — rectangle center: I = m(w²+h²)/12,  parallel axis: I = I_c + m d²
    I_top_center = m_top * (W**2 + top_t**2) / 12
    I_top = I_top_center + m_top * (y_top - y_com)**2
    I_stem_center = m_stem * (stem_w**2 + stem_h**2) / 12
    I_stem = I_stem_center + m_stem * (y_stem - y_com)**2
    inertia = I_top + I_stem

    # T polygon boundary (coordinates before COM correction)
    vertices = np.array([
        [-W/2,      H/2],          [ W/2,      H/2],
        [ W/2,      H/2 - top_t],  [ stem_w/2, H/2 - top_t],
        [ stem_w/2, -H/2],         [-stem_w/2, -H/2],
        [-stem_w/2, H/2 - top_t],  [-W/2,      H/2 - top_t],
    ], dtype=float)

    vertices[:, 1] -= y_com        # body local origin = center of mass
    return vertices, inertia

### 확인: 관성모멘트를 수치적으로 검증

정의 $I=\int r^2\,dm$ 을 Monte Carlo로 근사: T 내부에 균일하게 점을 뿌리면 각 점의 질량은 $m/N$ 이므로
$I \approx \frac{m}{N}\sum_{k=1}^{N}\lVert\mathbf r_k\rVert^2$ (COM이 원점이므로 $\mathbf r_k$ = 점의 좌표).

In [ ]:
vertices, inertia = make_T_geometry(mass=1.0)

rng = np.random.default_rng(0)
W, H, stem_w, top_t = 1.8, 1.8, 0.5, 0.5
y_shift = vertices[0, 1] - H / 2          # = -y_com (vertices were shifted by this)

pts = rng.uniform([-W / 2, -H / 2], [W / 2, H / 2], size=(400_000, 2))
in_top = pts[:, 1] >= H / 2 - top_t
in_stem = np.abs(pts[:, 0]) <= stem_w / 2
inside = pts[in_top | in_stem]
inside[:, 1] += y_shift                     # same COM-centred frame as `vertices`

print("COM of samples (should be ~0):", inside.mean(axis=0).round(4))
print("I  parallel-axis :", round(inertia, 5))
print("I  Monte Carlo   :", round(float(np.mean(np.sum(inside**2, axis=1))), 5))

## 4. Geometry helpers

**4.1 Point → segment 최소거리** — edge $A\rightarrow B$, circle center $P$ 에 대해
$t=\frac{(P-A)\cdot(B-A)}{\lVert B-A\rVert^2}$ 를 $[0,1]$ 로 clamp 하면 $\boxed{Q=A+t\,(B-A)}$. 모든 edge 중 가장 작은 $\lVert P-Q\rVert$ 를 찾습니다.

**4.2 Point in polygon (ray casting)** — $P$ 에서 $+x$ 방향 반직선이 edge와 교차한 횟수가 홀수면 내부, 짝수면 외부. **concave polygon**(T)에도 동작.

In [ ]:
# ============================================================
# 3. Geometry helper
# ============================================================
def closest_point_segment(p, a, b):
    """point p에서 line segment a-b로의 가장 가까운 점."""
    ab = b - a
    denominator = np.dot(ab, ab)
    if denominator < 1e-12:
        return a.copy()
    t = np.clip(np.dot(p - a, ab) / denominator, 0.0, 1.0)
    return a + t * ab

def point_in_polygon(p, vertices):
    """Ray casting algorithm. p가 concave polygon 내부에 있는지 확인."""
    x, y = p
    inside = False
    j = len(vertices) - 1
    for i in range(len(vertices)):
        xi, yi = vertices[i]
        xj, yj = vertices[j]
        if (yi > y) != (yj > y):                                   # crossing
            x_intersection = (xj - xi) * (y - yi) / (yj - yi) + xi
            if x < x_intersection:
                inside = not inside
        j = i
    return inside

## 5. Circle vs T polygon collision detection

**5.1 World → body local frame** — T가 회전해 있어도 항상 *고정된 T와 circle의 관계*만 보도록 circle center를 T local로 옮깁니다:
$\boxed{\mathbf p_C^{\text{local}} = R^T(\theta)\,(\mathbf p_C-\mathbf p_G)}$ (robotics / physics simulation의 단골 변환).

**5.2 Collision 조건** — radius $R$, boundary까지 최소거리 $d$ 이면 $\boxed{d<R}\Rightarrow$ collision, $\boxed{\delta=R-d}$ (penetration depth).

**5.3 Contact normal** — circle center $P$ → 가장 가까운 boundary 점 $Q$ (= pusher → T, T가 받는 normal impulse 방향):
$\boxed{\mathbf n=\frac{Q-P}{\lVert Q-P\rVert}}$, local normal은 $R(\theta)$ 를 곱해 world로 되돌림.
circle center가 T **내부**까지 들어간 비정상 상황이면 방향을 뒤집고 $\delta = R + d$.

```text
    │  Q ●←----- d -----○ P        (T boundary 왼쪽 면, P = circle center)
```

In [ ]:
# ============================================================
# 4. Circle vs T polygon collision
# ============================================================
def circle_vs_T(circle_pos, radius, body_pos, body_angle, vertices):
    """
    return: contact_point, normal, penetration
    normal은 pusher -> T 방향. 즉 T가 받는 normal impulse 방향.
    """
    R = rotation_matrix(body_angle)

    # World circle coordinate -> T local coordinate
    circle_local = R.T @ (circle_pos - body_pos)

    # Polygon의 모든 edge를 조사하여 circle center와 가장 가까운 boundary point 찾기
    best_point = None
    best_dist2 = float("inf")
    N = len(vertices)
    for i in range(N):
        a, b = vertices[i], vertices[(i + 1) % N]
        q = closest_point_segment(circle_local, a, b)
        delta = q - circle_local
        dist2 = np.dot(delta, delta)
        if dist2 < best_dist2:
            best_dist2 = dist2
            best_point = q
    distance = math.sqrt(best_dist2)

    # circle center가 T 내부인가?
    inside = point_in_polygon(circle_local, vertices)

    # circle outside이고 boundary까지 거리가 radius보다 크면 collision 없음
    if not inside and distance >= radius:
        return None

    # Normal direction
    if distance > 1e-12:
        direction_to_boundary = (best_point - circle_local) / distance
    else:
        direction_to_boundary = np.array([1.0, 0.0])

    if inside:
        # 매우 깊게 penetration한 비정상 상황 → polygon을 circle로부터 밖으로 밀기 위한 방향
        normal_local = -direction_to_boundary
        penetration = radius + distance
    else:
        # circle -> polygon
        normal_local = direction_to_boundary
        penetration = radius - distance

    # local normal -> world normal
    normal_world = R @ normal_local
    normal_world /= (np.linalg.norm(normal_world) + 1e-12)

    # contact point: polygon boundary 위의 nearest point
    contact_world = body_pos + R @ best_point

    return (contact_world, normal_world, penetration)

### 확인: collision detection 동작

T를 원점에 두고 circle을 몇 군데 놓아 봅니다. 같은 위치라도 T를 $45^\circ$ 돌리면 결과가 달라지는 것도 확인합니다.

In [ ]:
radius = 0.18
for label, c, ang in [
    ("left of stem, touching", np.array([-0.40, -0.30]), 0.0),
    ("far away",               np.array([-1.50,  0.00]), 0.0),
    ("same point, T at 45 deg", np.array([-0.40, -0.30]), math.pi / 4),
]:
    hit = circle_vs_T(c, radius, np.zeros(2), ang, vertices)
    if hit is None:
        print(f"{label:26s} -> no contact")
    else:
        q, n, pen = hit
        print(f"{label:26s} -> contact {q.round(3)}, normal {n.round(3)}, penetration {pen:.3f}")

## 6. Dynamic state

- `TBody` : dynamic body — 위치 $\mathbf x_G$, 각도 $\theta$, 속도 $\mathbf v_G$, 각속도 $\omega$, 질량 $m$, 관성모멘트 $I$
- `Pusher` : **kinematic body** — PD controller가 속도를 정하고, 충돌 impulse를 받아도 속도가 변하지 않습니다.

In [ ]:
# ============================================================
# 5. Dynamic state
# ============================================================
@dataclass
class TBody:
    pos: np.ndarray
    angle: float
    vel: np.ndarray
    omega: float
    mass: float
    inertia: float
    vertices: np.ndarray

@dataclass
class Pusher:
    pos: np.ndarray
    vel: np.ndarray
    radius: float

## 7. Collision impulse — PushT physics의 핵심

| 단계 |---------------------식----------------------| 의미 |
|---|---------------------|---|
| 7.1 Contact point 속도 | $\boxed{\mathbf v_C=\mathbf v_G+\boldsymbol\omega\times\mathbf r},\ \ \mathbf r = \mathbf x_{\text{contact}}-\mathbf x_G$ | Meriam의 relative velocity. COM 속도와 contact 속도는 다르다 |
| 7.2 상대속도 | $\boxed{\mathbf v_{\text{rel}}=\mathbf v_C-\mathbf v_P},\ \ \boxed{v_n=\mathbf v_{\text{rel}}\cdot\mathbf n}$ | $\mathbf n$ = pusher → T 이므로 $v_n<0$ 이면 **서로 접근** 중 |
| 7.3 Restitution | $\boxed{v_n^{+}=-e\,v_n^{-}}$ | PushT는 거의 비탄성 ($e\approx0$) → 충돌 뒤 $v_n^{+}\approx0$, 더 이상 파고들지 않음 |
| 7.4 Linear impulse | $m\mathbf v^{+}=m\mathbf v^{-}+\mathbf J\ \Rightarrow\ \boxed{\Delta\mathbf v_G=\mathbf J/m}$ | $\mathbf J = j\,\mathbf n$ |
| 7.4 Angular impulse | $I\,\Delta\omega=\mathbf r\times\mathbf J\ \Rightarrow\ \boxed{\Delta\omega=(\mathbf r\times\mathbf J)/I}$ | **pusher가 T 옆부분을 밀면 T가 회전하는 이유** |
| 7.5 Effective inverse mass | $\boxed{K_n=\frac1m+\frac{(\mathbf r\times\mathbf n)^2}{I}},\ \ \boxed{j=-\frac{(1+e)\,v_n}{K_n}}$ | impulse가 $\mathbf v_G$ 와 $\omega$ 를 **함께** 바꾸므로 분모에 회전항 |

**7.5 해석** — contact가 COM 근처면 $\mathbf r\times\mathbf n\approx0$, $K_n\approx 1/m$ → 대부분 translation. 멀리 밀면 $\lvert\mathbf r\times\mathbf n\rvert$ 가 커져 impulse 일부가 회전으로.
$\boxed{\text{same push}+\text{different contact position}\Rightarrow\text{different translation / rotation}}$ — PushT task의 본질 (실험 8에서 확인).

**7.6 Pusher도 dynamic body라면?** $K_n=\frac1{m_A}+\frac1{m_B}+\frac{(\mathbf r_A\times\mathbf n)^2}{I_A}+\frac{(\mathbf r_B\times\mathbf n)^2}{I_B}$.
kinematic pusher는 $1/m_P = 0,\ 1/I_P=0$ 으로 취급하므로 T의 항만 남습니다.

**7.7 Friction impulse $j_t$** — normal만 쓰면 pusher가 표면에서 너무 쉽게 미끄러짐. tangent $\boxed{\mathbf t=(-n_y,\ n_x)}$, $v_t=\mathbf v_{\text{rel}}\cdot\mathbf t$ 를 없애는 impulse
$j_t^{*}=-\frac{v_t}{K_t},\ K_t=\frac1m+\frac{(\mathbf r\times\mathbf t)^2}{I}$ 를 **Coulomb friction** $\boxed{\lvert j_t\rvert\le\mu\,j_n}$ 로 clamp.

**7.8 Position correction** — discrete simulation은 $t\rightarrow t+\Delta t$ 로 점프하므로 `○ | T` → `○|T` 처럼 이미 겹친 상태가 생깁니다.
velocity impulse만으로는 겹침이 남으므로 normal 방향으로 직접 밀어냅니다: $\Delta\mathbf x_G=\beta\,\max(\delta-\text{slop},\,0)\,\mathbf n$.

In [ ]:
# ============================================================
# 6. Collision impulse
# ============================================================
def resolve_impulse(body, pusher, contact, restitution=0.0, friction=0.4):
    contact_point, normal, penetration = contact

    # Contact position relative to T center of mass
    r = contact_point - body.pos

    # velocity of contact point on T:  v_c = v_G + omega x r
    v_contact_T = body.vel + omega_cross_r(body.omega, r)

    # relative velocity = T contact velocity - pusher velocity,  and its normal component
    v_relative = v_contact_T - pusher.vel
    v_normal = float(np.dot(v_relative, normal))

    j_normal = 0.0
    j_tangent = 0.0

    if v_normal < 0:                    # vn < 0 means approaching
        # Effective inverse mass  K = 1/m + (r x n)^2 / I   (pusher = kinematic -> inverse mass = 0)
        K_normal = 1.0 / body.mass + cross2(r, normal)**2 / body.inertia

        # normal impulse magnitude  j = -(1+e) vn / K
        j_normal = -(1.0 + restitution) * v_normal / K_normal
        impulse_normal = j_normal * normal

        body.vel += impulse_normal / body.mass                          # Δv = J / m
        body.omega += cross2(r, impulse_normal) / body.inertia          # Δω = (r x J) / I

        # ---------------- friction impulse ----------------
        # normal impulse 적용 후 새로운 contact velocity 계산
        v_contact_T = body.vel + omega_cross_r(body.omega, r)
        v_relative = v_contact_T - pusher.vel

        tangent = np.array([-normal[1], normal[0]])
        v_tangent = float(np.dot(v_relative, tangent))
        K_tangent = 1.0 / body.mass + cross2(r, tangent)**2 / body.inertia
        j_tangent_free = -v_tangent / K_tangent

        # Coulomb friction  |Jt| <= mu * Jn
        j_tangent = float(np.clip(j_tangent_free, -friction * j_normal, friction * j_normal))
        impulse_tangent = j_tangent * tangent

        body.vel += impulse_tangent / body.mass
        body.omega += cross2(r, impulse_tangent) / body.inertia

    # Position correction: 이미 물체가 조금 겹친 경우 바로 분리해준다.
    slop = 1e-4
    beta = 0.8
    correction = beta * max(penetration - slop, 0.0)
    body.pos += correction * normal

    return (j_normal, j_tangent)

## 8. 실험: 같은 push, 다른 contact 높이

7.5의 주장을 숫자로 확인합니다. 정지한 T의 **stem 왼쪽 면** ($x=-0.25$) 을 pusher가 같은 속도 $\mathbf v_P=(1,0)$ 로 밀고 접촉 높이 $y$ 만 바꿉니다 ($\mu=0$, $e=0$).
$\mathbf n=(1,0)$, $\mathbf r=(-0.25,\,y)$ 이므로 $\mathbf r\times\mathbf n=-y$ 이고
$K_n=\frac1m+\frac{y^2}{I},\ j=\frac{1}{K_n},\ \Delta v_x=\frac{j}{m},\ \Delta\omega=\frac{-y\,j}{I}$.

- $y=0$ (COM 높이) : 회전 없이 순수 translation, $\Delta v_x = 1$  ·  $\lvert y\rvert$ 가 클수록 : $j$ 와 $\Delta v_x$ 는 줄고, 그만큼 $\lvert\Delta\omega\rvert$ 가 커짐

In [ ]:
stem_left = -0.25
y_bottom = vertices[:, 1].min()             # bottom of the stem (COM frame)
y_stem_top = vertices[3, 1]                 # where the stem meets the top bar

print(f"{'contact y':>10s} {'K_n':>8s} {'j_n':>8s} {'dv_x':>8s} {'d_omega':>9s}")
for y in np.sort(np.r_[np.linspace(y_bottom + 0.05, y_stem_top - 0.05, 5), 0.0]):   # include y = 0 (COM height)
    body = TBody(pos=np.zeros(2), angle=0.0, vel=np.zeros(2), omega=0.0,
                 mass=1.0, inertia=inertia, vertices=vertices)
    pusher = Pusher(pos=np.array([stem_left - 0.18, y]), vel=np.array([1.0, 0.0]), radius=0.18)
    contact = (np.array([stem_left, y]), np.array([1.0, 0.0]), 0.0)   # point, normal, penetration
    K_n = 1.0 / body.mass + cross2(contact[0] - body.pos, contact[1]) ** 2 / body.inertia
    jn, _ = resolve_impulse(body, pusher, contact, restitution=0.0, friction=0.0)
    print(f"{y:10.3f} {K_n:8.3f} {jn:8.3f} {body.vel[0]:8.3f} {body.omega:9.3f}")

## 9. Mini PushT simulator

한 번의 `step(action)` 은 substep마다 아래 A–D를 반복합니다.

| | 단계 | 식 |
|---|---|---|
| **A** | Pusher PD control (속도는 `max_speed` 로 제한) | $\mathbf a_P = K_p(\mathbf x^{*}-\mathbf x_P) - K_d\mathbf v_P,\ \ \mathbf v_P \leftarrow \mathbf v_P+\mathbf a_P\Delta t,\ \ \mathbf x_P \leftarrow \mathbf x_P+\mathbf v_P\Delta t$ |
| **B** | T damping (바닥 마찰 흉내, 지수 감쇠) | $\mathbf v_G \leftarrow \mathbf v_G\,e^{-c_v\Delta t},\ \ \omega \leftarrow \omega\,e^{-c_\omega\Delta t}$ |
| **C** | T 적분 (semi-implicit Euler) | $\mathbf x_G \leftarrow \mathbf x_G+\mathbf v_G\Delta t,\ \ \theta \leftarrow \theta+\omega\Delta t$ |
| **D** | Contact solver | `circle_vs_T` → `resolve_impulse` 를 최대 3번 반복 |

왜 여러 번? 한 번의 impulse로 충돌이 완벽히 해결되지 않는 경우가 많기 때문 (friction, rotation, penetration, 여러 contact, corner contact).
Pymunk, Box2D도 훨씬 정교한 **iterative constraint solver** 를 사용합니다.

In [ ]:
# ============================================================
# 7. Mini PushT simulator
# ============================================================
class MiniPushT:

    def __init__(self):
        mass = 1.0
        vertices, inertia = make_T_geometry(mass=mass)

        self.T = TBody(pos=np.array([0.0, 0.0]), angle=0.0, vel=np.zeros(2), omega=0.0,
                       mass=mass, inertia=inertia, vertices=vertices)

        # pusher starts close to T so that one control step can create collision
        self.pusher = Pusher(pos=np.array([-1.15, 0.30]), vel=np.zeros(2), radius=0.18)

        # simulation frequency
        self.control_dt = 0.1
        self.physics_dt = 0.01
        self.substeps = int(self.control_dt / self.physics_dt)

        # pusher PD controller
        self.kp, self.kd, self.max_speed = 80.0, 18.0, 3.0

        # T damping
        self.linear_damping, self.angular_damping = 1.5, 1.5

        # collision properties
        self.restitution, self.friction = 0.0, 0.4

    def observation(self):
        return np.array([self.pusher.pos[0], self.pusher.pos[1],
                         self.T.pos[0], self.T.pos[1], self.T.angle])

    def step(self, action):
        """action: [target_x, target_y].  One RL/control step."""
        target = np.asarray(action, dtype=float)
        collision_impulses = []

        # One control step = multiple physics steps
        for _ in range(self.substeps):
            dt = self.physics_dt

            # A. Pusher PD control
            acceleration = self.kp * (target - self.pusher.pos) - self.kd * self.pusher.vel
            self.pusher.vel += acceleration * dt
            speed = np.linalg.norm(self.pusher.vel)                 # speed limit
            if speed > self.max_speed:
                self.pusher.vel *= self.max_speed / speed
            self.pusher.pos += self.pusher.vel * dt                 # pusher integration

            # B. T damping
            self.T.vel *= math.exp(-self.linear_damping * dt)
            self.T.omega *= math.exp(-self.angular_damping * dt)

            # C. Integrate T
            self.T.pos += self.T.vel * dt
            self.T.angle += self.T.omega * dt

            # D. Contact solver — repeated a few times for stability
            for _ in range(3):
                contact = circle_vs_T(self.pusher.pos, self.pusher.radius,
                                      self.T.pos, self.T.angle, self.T.vertices)
                if contact is None:
                    break
                jn, jt = resolve_impulse(self.T, self.pusher, contact,
                                         restitution=self.restitution, friction=self.friction)
                collision_impulses.append((jn, jt))

        return (self.observation(), collision_impulses)

## 10. Test: one control step

Pusher는 T의 왼쪽 ($x=-1.15$, $y=0.30$) 에서 시작하고, target을 오른쪽 ($x=0$) 으로 줍니다. $y=0.30$ 은 COM보다 위이므로 push와 함께 회전도 생겨야 합니다.

In [ ]:
# ============================================================
# 8. Test one step
# ============================================================
sim = MiniPushT()
print("Before:")
print("obs =", sim.observation())
print("T velocity =", sim.T.vel)
print("T omega =", sim.T.omega)

action = np.array([0.0, 0.30])          # pusher target: move to the right
obs, impulses = sim.step(action)

print("\nAfter one step:")
print("obs =", obs)
print("T velocity =", sim.T.vel)
print("T omega =", sim.T.omega)
print("impulses =", impulses)

## 11. 실험: 여러 step rollout과 시각화

같은 target을 15 step (1.5 s) 동안 유지하면서 T와 pusher를 그립니다. 색이 진해질수록 나중 시점 — T가 오른쪽으로 밀리면서 동시에 회전하는 것이 보여야 합니다.

In [ ]:
import matplotlib.pyplot as plt

sim = MiniPushT()
frames = [(sim.T.pos.copy(), sim.T.angle, sim.pusher.pos.copy())]
for _ in range(15):
    sim.step(np.array([0.6, 0.30]))
    frames.append((sim.T.pos.copy(), sim.T.angle, sim.pusher.pos.copy()))

fig, ax = plt.subplots(figsize=(7, 5))
for k, (pos, ang, ppos) in enumerate(frames):
    if k % 3:
        continue
    shade = 0.25 + 0.75 * k / (len(frames) - 1)
    world = pos + vertices @ rotation_matrix(ang).T
    ax.fill(*np.vstack([world, world[:1]]).T, fc=(0.85, 0.93, 0.99, 0.35), ec=(0.15, 0.45, 0.72, shade), lw=2)
    ax.add_patch(plt.Circle(ppos, sim.pusher.radius, color=(0.91, 0.47, 0.13, shade)))
    ax.plot(*pos, "k.", ms=4)
path = np.array([f[2] for f in frames])
ax.plot(path[:, 0], path[:, 1], "--", color="0.5", lw=1, label="pusher path")
ax.set_aspect("equal")
ax.set_title("Mini PushT: 15 control steps toward target (0.6, 0.3)")
ax.legend(loc="lower right")
plt.show()

print("final T position:", sim.T.pos.round(3), " angle (deg):", round(math.degrees(sim.T.angle), 1))

## 12. 정리 — PushT 1 step을 수식으로

| # | 단계 | ---------------------------식 ---------------------------|
|---|---|---|
| 1 | Action | $a_t=(x^{*},y^{*})$ |
| 2 | Pusher controller | $\mathbf a_P=K_p(\mathbf x^{*}-\mathbf x_P)-K_d\mathbf v_P$ |
| 3 | Collision detection | $(\text{circle},\,T)\rightarrow(\mathbf q,\,\mathbf n,\,\delta)$ |
| 4 | Contact / normal velocity | $\mathbf v_C=\mathbf v_G+\boldsymbol\omega\times\mathbf r,\ \ v_n=(\mathbf v_C-\mathbf v_P)\cdot\mathbf n$ |
| 5 | Impulse | $j_n=-\frac{(1+e)\,v_n}{K_n},\ \ \mathbf J_n=j_n\mathbf n$ |
| 6 | Velocity update | $\mathbf v_G\leftarrow\mathbf v_G+\frac{\mathbf J_n}{m},\ \ \omega\leftarrow\omega+\frac{\mathbf r\times\mathbf J_n}{I}$ |
| 7 | Integration | $\mathbf x_G\leftarrow\mathbf x_G+\mathbf v_G\Delta t,\ \ \theta\leftarrow\theta+\omega\Delta t$ |

### 가장 중요한 4개의 식

$$
\boxed{\mathbf v_C=\mathbf v_G+\boldsymbol\omega\times\mathbf r}\qquad
\boxed{j=-\frac{(1+e)\,v_n}{1/m+(\mathbf r\times\mathbf n)^2/I}}\qquad
\boxed{\mathbf v_G^{+}=\mathbf v_G^{-}+\frac{\mathbf J}{m}}\qquad
\boxed{\omega^{+}=\omega^{-}+\frac{\mathbf r\times\mathbf J}{I}}
$$

이 네 식이 collision detection에서 구한 `(contact point, normal)` 을 실제 T-block의 translation과 rotation으로 연결하는 다리입니다.

이 예제는 의도적으로 단순화했습니다. Pymunk / Box2D 수준으로 가면 multiple contact manifold, warm starting,
static/dynamic friction, constraint stabilization, continuous collision detection 등이 추가됩니다.
하지만 PushT의 물리를 이해하고 작은 simulator를 직접 만드는 단계에서는 이 정도가 가장 좋은 출발점입니다.